In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# Final Production Deployment
# ========================================

from pyspark.sql.functions import *

try:

    # ---------------------------------------
    # Optimize Delta Tables
    # ---------------------------------------

    spark.sql(f"""
    OPTIMIZE delta.`{gold_path}/claims_analytics`
    """)

    spark.sql(f"""
    OPTIMIZE delta.`{gold_path}/hospital_revenue_analytics`
    """)

    print("Delta Optimization Completed")

    # ---------------------------------------
    # Validation Testing
    # ---------------------------------------

    bronze_count = (
        spark.read.format("delta")
        .load(f"{bronze_path}/patients")
        .count()
    )

    silver_count = (
        spark.read.format("delta")
        .load(f"{silver_path}/patients_clean")
        .count()
    )

    print(f"Bronze Count : {bronze_count}")
    print(f"Silver Count : {silver_count}")

    if bronze_count >= silver_count:
        print("Validation Passed")
    else:
        print("Validation Failed")

    # ---------------------------------------
    # Delta History
    # ---------------------------------------

    print("Displaying Delta History")

    display(
        spark.sql(f"""
        DESCRIBE HISTORY delta.`{silver_path}/patients_clean`
        """)
    )

    # ---------------------------------------
    # Time Travel
    # ---------------------------------------
    # I implemented Delta Lake Time Travel using versionAsOf. During testing, older versions were unavailable because the required historical files

    try:

        previous_version_df = (
            spark.read.format("delta")
            .option("versionAsOf", 0)
            .load(f"{silver_path}/patients_clean")
        )

        previous_version_df.show()

        print("Time Travel Successful")

    except Exception as e:

        print("Time Travel Failed")
        print(f"Reason: {str(e)}")

        print("""
Historical files required for Version 0 are not available.

This may happen due to:
1. Delta retention policy
2. VACUUM operation
3. Overwrite operations performed during development
        """)

    print("Production Deployment Completed Successfully")

except Exception as e:

    print(f"Deployment Failed : {str(e)}")